# 260313 LangChain 입문 - ChatOpenAI · PromptTemplate · LCEL 파이프

**Day 4 · 2026-03-13 (금)** · 1주차 W1 · API & Chain 기초

---

## 오늘의 학습 목표
- `langchain`이 왜 필요한지 감 잡기 (모델 교체/재사용/체이닝)
- `ChatOpenAI` + `SystemMessage` / `HumanMessage` / `AIMessage` 메시지 클래스 사용
- `invoke` / `stream` / `batch` 세 가지 실행 모드 비교
- `bind()` 로 모델 파라미터·응답 포맷 고정하기
- `ChatPromptTemplate` + `partial()` 로 재사용 가능한 프롬프트 만들기
- LCEL 파이프 연산자 `|` 로 **prompt → llm → parser** 체인 구성
- `StrOutputParser` / `JsonOutputParser` / `PydanticOutputParser` 비교

> 비유: 어제까지의 `openai` SDK가 **재료를 한 접시에 담아 내놓는 방식**이었다면, 오늘의 LangChain은 **컨베이어 벨트**다. 재료(prompt) → 요리사(llm) → 접시(parser)가 파이프로 연결돼 한 번만 세팅하면 다음부턴 같은 벨트에 재료만 바꿔 넣으면 된다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w1_api_and_chain/llm_260313_openAi_langChain.ipynb)

## 0. 환경 준비

LangChain은 여러 패키지로 쪼개져 있다.
- `langchain` : 핵심 패키지 (체인, 프롬프트, 파서 등 뼈대)
- `langchain-openai` : OpenAI 모델 래퍼(ChatOpenAI 등)
- `langchain-community` : 서드파티 로더·툴 모음 (웹·파일·DB 로더 등)

> 비유: LangChain은 **종합 가전제품 브랜드**. 본체(langchain) 외에 옵션(openai 어댑터, community 모듈)을 따로 사야 완전체가 된다.

In [ ]:
# LangChain 3종 세트 + pydantic + dotenv 설치
!pip install -q langchain langchain-openai langchain-community python-dotenv pydantic

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 1. LangChain이 해결하는 문제

1. **모델 교체가 쉽다**: `ChatOpenAI` 대신 `ChatAnthropic`, `ChatGoogleGenerativeAI` 로 한 줄만 바꾸면 됨.
2. **체이닝(chain)**: prompt → model → output parser를 파이프로 엮어 한 줄로 실행.
3. **유틸**: 문서 로더, 벡터스토어 연동, 메모리 등 RAG 파이프라인용 도구가 풍부.

기본 호출 패턴은 어제와 거의 같다 - 단, `client.chat.completions.create(...)` 대신 `llm.invoke(...)` 를 쓴다.

In [ ]:
# ChatOpenAI - 어제의 OpenAI() 클라이언트에 해당하는 LangChain 래퍼
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4o-mini',   # 저렴한 주력 모델
    temperature=0.7,       # 약간의 창의성 허용
    api_key=api_key,       # 위에서 로드한 키 전달
)

In [ ]:
# invoke: 가장 기본 실행 - 문자열 하나를 그대로 넣을 수도 있다
response = llm.invoke('LangChain을 한 문장으로 설명해 주세요.')
response  # AIMessage 객체가 반환됨

In [ ]:
# 타입과 실제 답변 확인 - content 속성에 텍스트가 들어 있다
print(type(response))       # langchain_core.messages.ai.AIMessage
print(response.content)     # 실제 답변 텍스트

## 2. 메시지 클래스 - SystemMessage / HumanMessage / AIMessage

어제는 `{'role': 'system', 'content': ...}` 같은 **딕셔너리**였지만, LangChain은 전용 클래스를 제공한다.
- `SystemMessage` ↔ role='system'
- `HumanMessage` ↔ role='user'
- `AIMessage` ↔ role='assistant'

> 비유: 같은 **편지**인데 라벨(클래스)만 바꿔 단 느낌. 내용물은 그대로지만 봉투가 깔끔해진다.

In [ ]:
# SystemMessage + HumanMessage 조합으로 페르소나 지정
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

persona = 'MLOps Leader'
question = 'Langchain 배포할 때, 고려해야하는 리스크를 세가지 알려주세요.'

messages = [
    SystemMessage(content=f'당신은 {persona}입니다. 간결하게 대답해 주세요.'),
    HumanMessage(content=question),
]
response = llm.invoke(messages)
response

In [ ]:
# 토큰 사용량 등 메타데이터 확인 - response_metadata에 모델명/usage가 들어 있다
response.response_metadata

In [ ]:
# 실습: 페르소나·질문을 받아 답변+토큰까지 리턴하는 헬퍼 함수
def ask_expert(persona, question):
    messages = [
        SystemMessage(content=f'당신은 {persona}입니다. 실무 위주로 간단히 대답해 주세요.'),
        HumanMessage(content=question),
    ]
    response = llm.invoke(messages)
    usage = response.response_metadata.get('token_usage', {})
    return {'answer': response.content, 'token_usage': usage}

# 테스트 호출
result = ask_expert('데이터 과학자', 'LangChain 프로젝트에 로깅을 추가하는 이유는?')
print(result['answer'])
print('---')
print(result['token_usage'])

## 3. `invoke` / `stream` / `batch` - 세 가지 실행 모드

LangChain `Runnable` 인터페이스가 공통으로 지원하는 세 가지 실행법.
- **invoke(x)** : 하나 넣고 하나 받기 (동기).
- **stream(x)** : 한 번 넣고 **청크 단위로** 조각조각 받기.
- **batch([x1, x2, ...])** : 여러 개를 **병렬로** 한 번에 처리.

> 비유: 카페 주문. **invoke = 한 잔 주문**, **stream = 리필 물 조금씩 따르기**, **batch = 단체 주문 한 번에 들고 가기**.

In [ ]:
# 기본 invoke - 메시지 리스트로 질문
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.7, api_key=api_key)
messages = [
    SystemMessage(content='당신은 친절한 AI 튜터입니다. 간결하게 답변하세요'),
    HumanMessage(content='Langchain이란 무엇인가요?'),
]
llm.invoke(messages)

In [ ]:
# stream - 청크가 도착하는 대로 출력 (체감 반응 속도 up)
for chunk in llm.stream('프렌즈의 주인공은 누구인가요?'):
    print(chunk.content, end='', flush=True)

In [ ]:
# batch - 질문 3개를 병렬 처리 (3회 순차 호출보다 훨씬 빠름)
questions = [
    '모니카에 대해 간단하게 설명해 줘.',
    '피비의 특징에 대해 간단하게 설명해줘',
    '챈들러 배우에 대해 간단하게 설명해줘',
]
results = llm.batch(questions)

for result in results:
    print(result.content)
    print('---')

In [ ]:
# batch 고급 - 각 입력이 서로 다른 system 프롬프트를 갖는 경우
batch_inputs = [
    [
        SystemMessage(content='당신은 데이터 과학자입니다. 답변은 두 항목으로 요약.'),
        HumanMessage(content='LangChain 프로젝트에 로깅을 추가하는 이유는?'),
    ],
    [
        SystemMessage(content='당신은 프롬프트 엔지니어입니다. 핵심만 말하세요.'),
        HumanMessage(content='LLM 호출을 프로파일링할 때 살펴볼 지표 세 가지는?'),
    ],
    [
        SystemMessage(content='당신은 교육 책임자입니다. 초보자에게 설명하세요.'),
        HumanMessage(content='체인 디버깅 세션을 어떻게 구성할까?'),
    ],
]

batch_results = llm.batch(batch_inputs)
for r in batch_results:
    print(r.content)
    print('---')

In [ ]:
# stream을 리스트에 모은 뒤 문자열로 합치기 - 걸린 시간도 측정
import time

stream_llm = llm.bind(temperature=0, max_tokens=80)  # bind는 곧 배운다
stream_chunks = []

start = time.perf_counter()
for chunk in stream_llm.stream(messages):
    stream_chunks.append(chunk.content)
elapsed = time.perf_counter() - start

print('청크 개수:', len(stream_chunks))
print('소요 시간:', round(elapsed, 2), '초')
print('합친 답변:', ''.join(stream_chunks))

## 4. `bind()` - 런타임 파라미터/옵션 고정

모델 호출마다 같은 옵션(temperature, max_tokens, response_format ...)을 지겹도록 주기 싫을 때 `llm.bind(...)`로 **미리 묶어** 새 llm을 만든다.

- 파라미터 고정: `llm.bind(temperature=0, max_tokens=80)`
- **응답 포맷을 JSON으로 강제**: `llm.bind(response_format={'type': 'json_object'})`

> 비유: 자주 쓰는 매크로를 **단축키 하나에 할당**해 두는 것. 매번 풀세팅 안 해도 된다.

주의: JSON 포맷 강제는 **system/user 메시지 안에 "JSON"이라는 단어가 꼭 포함**돼야 동작한다.

In [ ]:
# JSON 출력 강제 - 프롬프트에 'json' 단어가 있어야 함
llm_json = llm.bind(response_format={'type': 'json_object'})

response = llm_json.invoke([
    SystemMessage(content='당신은 미국 드라마 프렌즈 마니아 입니다. 질문에 대해 json 형식으로 답변해 주세요.'),
    HumanMessage(content='프렌즈의 주인공은 무엇인가요?'),
])
print(response.content)

In [ ]:
# content는 문자열이지만 JSON 규격이므로 json.loads로 파싱 가능
import json
data = json.loads(response.content)
print(type(data))
print(json.dumps(data, ensure_ascii=False, indent=2))

## 5. `ChatPromptTemplate` - 프롬프트 재사용 템플릿

같은 프롬프트를 질문만 바꿔 여러 번 쓰고 싶을 때, 매번 f-string 조립하지 말고 템플릿 객체로 만들어 둔다.
- 중괄호 `{변수}` 자리를 남겨 두고 나중에 `invoke({...})` 로 채움.
- `input_variables` 속성으로 어떤 변수가 필요한지 확인 가능.
- `partial(...)` 로 **일부 변수를 미리** 채워 고정할 수 있음.

> 비유: **이력서 양식**. 이름/경력란만 빈칸으로 두고, 지원할 때마다 그 칸만 채워 넣는 식.

In [ ]:
# 프롬프트 템플릿 정의 - role, style, question 3개 슬롯
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {role} 전문가입니다. {style}로 답변하세요.'),
    ('human',  '{question}'),
])

# 어떤 변수가 필요한지 확인
print('input_variables:', prompt.input_variables)

In [ ]:
# 템플릿에 값 채우기 - ChatPromptValue가 반환됨 (아직 모델 호출은 안 함)
results = prompt.invoke({
    'role': '파이썬',
    'style': '간결하게',
    'question': '파이썬에서 함수 쓰는 방법?',
})

# 메시지별 내용 출력
for msg in results.messages:
    print(f'[{msg.type}] {msg.content}')

In [ ]:
# partial() - role을 미리 고정. 이후엔 question만 넘기면 됨
prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {role} 전문가입니다. 핵심만 3줄로 답변해주세요'),
    ('human',  '{question}'),
])

partial_prompt = prompt.partial(role='파이썬')  # role만 먼저 채움
print('원본 :', prompt.input_variables)
print('부분 :', partial_prompt.input_variables)  # question만 남아야 함

In [ ]:
# LCEL 파이프 첫 사용 - partial_prompt | llm 형태로 체인 구성
# | 의 의미: 왼쪽의 출력이 오른쪽의 입력으로 흐른다 (리눅스 파이프와 동일)
response = (partial_prompt | llm).invoke({'question': '| 파이프 문법은 어떻게 사용하나요?'})
print(response.content)

## 6. LCEL 파이프 체인 - `prompt | llm | parser`

LangChain Expression Language(LCEL)의 가장 익숙한 패턴.

```python
chain = prompt | llm | parser
chain.invoke({...})
```

리눅스 파이프(`ls | grep ...`) 문법을 그대로 빌려온 것. 왼쪽에서 오른쪽으로 데이터가 흐른다: **변수 → 프롬프트 완성 → 모델 답변(AIMessage) → 파서(문자열/딕셔너리/객체)**.

> 비유: **컵라면 조리대**. 물(입력) → 포트(prompt) → 면(llm) → 그릇(parser). 한 번 세팅해두면 다음부턴 물만 부으면 끝.

In [ ]:
# 컨텍스트를 포함한 실전 프롬프트 - 두 개의 role로 같은 질문에 답변 비교
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ('system',
     '당신은 {role} 입니다. 아래 컨텍스트를 꼭 반영해서 액션 아이템을 제시해주세요.\n'
     '[Context]\n {context}\n'
     '- 답변은 불릿 3개 이하로 구성\n - 각 불릿은 동사로 시작'),
    ('human', '{question}'),
])

parser = StrOutputParser()  # AIMessage에서 content만 뽑아 str로 변환

# 같은 템플릿 + 다른 role 로 두 체인 구성 (partial로 role 고정)
pm_chain = prompt.partial(role='프로덕트 매니저') | llm | parser
ml_chain = prompt.partial(role='ML 리드')       | llm | parser

question = 'Langchain 워크샵을 90분 세션으로 구성하려 할 때, 어떤 활동이 필요할까요?'
context = '''-대상 : 사내 주니어 엔지니어 25명
- 장비 : 실습용 노트북 12대, HDMI 2개
- 목표 : Langchain 핵심 컴포넌트 맛보기 + 실습 페어링'''

# 두 페르소나의 답변 비교
pm_answer = pm_chain.invoke({'question': question, 'context': context})
ml_answer = ml_chain.invoke({'question': question, 'context': context})

print('=== [PM 관점] ===')
print(pm_answer)
print()
print('=== [ML 리드 관점] ===')
print(ml_answer)

## 7. 출력 파서(Output Parser) - 세 가지 주요 타입

모델 답변(`AIMessage`)을 **바로 쓸 수 있는 형태**로 변환하는 마지막 단계.

| Parser | 반환 타입 | 용도 |
|---|---|---|
| `StrOutputParser` | `str` | 단순 텍스트 응답 |
| `JsonOutputParser` | `dict` | 구조화된 JSON 응답 (pydantic 스키마 안내 가능) |
| `PydanticOutputParser` | Pydantic 객체 | 타입/범위 검증까지 필요한 경우 |

`llm.bind(response_format='json_object')` 와의 차이:
- **bind**는 **모델에게** JSON으로 답해달라고 요청 (포맷 강제, 하지만 완벽 보장은 아님).
- **parser**는 **답변을 받은 뒤 후처리**. 조합하면 더 안전하다.

> 비유: bind = **주문서에 'JSON으로 주세요' 써두기**, parser = **배달 온 음식을 그릇에 맞게 옮겨 담기**. 둘 다 하면 사고 확률이 낮아진다.

In [ ]:
# StrOutputParser - AIMessage.content를 그대로 str로 반환
prompt = ChatPromptTemplate.from_messages([
    ('human', '{topic}에 대해 한 줄로 설명해주세요'),
])

parser = StrOutputParser()
chain = prompt | llm | parser

response = chain.invoke({'topic': 'Langchain'})
print(response)
print('type:', type(response))  # str

In [ ]:
# JsonOutputParser + Pydantic 스키마 - 원하는 필드/타입을 명시
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class CityInfo(BaseModel):
    name: str        = Field(description='도시이름')
    country: str     = Field(description='국가')
    population: str  = Field(description='인구 (대략적)')
    famous_for: list[str] = Field(description='유명한 것 3가지')
    city_score: float     = Field(description='도시의 관광 점수')  # 100점 만점 기준

parser = JsonOutputParser(pydantic_object=CityInfo)

# 파서가 자동 생성하는 포맷 지시문 확인 - 프롬프트에 끼워 넣게 된다
print(parser.get_format_instructions()[:400], '...')

In [ ]:
# 체인 실행 - format_instructions를 프롬프트 변수로 주입
prompt = ChatPromptTemplate.from_messages([
    ('system', '정확한 정보를 json 형식으로 응답해주세요,\n{format_instructions}'),
    ('human',  '{city}에 대해서 알려주세요'),
])
chain = prompt | llm | parser

result = chain.invoke({
    'city': '서울',
    'format_instructions': parser.get_format_instructions(),
})

print('type:', type(result))  # dict
print(result)
print('name:', result['name'])
print('country:', result['country'])

In [ ]:
# PydanticOutputParser - 딕셔너리 대신 Pydantic 객체로 검증까지
from langchain_core.output_parsers import PydanticOutputParser

class BookReview(BaseModel):
    title: str    = Field(description='책 제목')
    author: str   = Field(description='저자')
    rating: int   = Field(description='평점 (1-5)', ge=1, le=5)  # 범위 검증
    summary: str  = Field(description='한줄요약')
    recommended: bool = Field(description='추천 여부')

parser = PydanticOutputParser(pydantic_object=BookReview)

prompt = ChatPromptTemplate.from_messages([
    ('system', '도서 리뷰를 작성해주세요.\n{format_instructions}'),
    ('human',  '{book} 책에 대한 리뷰를 작성해주세요'),
])

chain = prompt | llm | parser

result = chain.invoke({
    'book': '장발장',
    'format_instructions': parser.get_format_instructions(),
})

print('type:', type(result))  # BookReview 객체
print('title      :', result.title)
print('author     :', result.author)
print('rating     :', result.rating)
print('summary    :', result.summary)
print('recommended:', result.recommended)

## 정리 - 오늘의 요약

- [x] `ChatOpenAI`(+ `SystemMessage`/`HumanMessage`/`AIMessage`)로 어제와 동일한 호출을 LangChain 스타일로 재작성
- [x] `invoke` / `stream` / `batch` - 한 개, 스트리밍, 병렬 실행 3종 모드
- [x] `bind()`로 파라미터 또는 `response_format`(JSON) 고정
- [x] `ChatPromptTemplate` + `partial()` - 프롬프트 템플릿 재사용
- [x] LCEL 파이프 `prompt | llm | parser` - 한 줄 체인
- [x] `StrOutputParser` / `JsonOutputParser` / `PydanticOutputParser` - 출력 형태별 파서

**이번 주 실무 TIP**
- 포맷 강제는 bind + parser를 **함께** 쓰는 게 가장 안전.
- 값 범위(`ge=1, le=5`)까지 확실히 지켜야 한다면 `PydanticOutputParser`.
- 간단한 텍스트 응답이면 `StrOutputParser`만으로 충분.
- 체인을 재활용하고 싶을 때는 `partial()`로 일부 변수를 미리 고정.

내일(토)은 주말 프로젝트 시간 - 주택청약 FAQ 챗봇의 뼈대를 오늘 배운 체인으로 붙여 볼 차례다.